# Projektarbeit: Applied Neural Networks

**Projektname:** Kleiderschrank sortieren

**Autoren:** Daria Lukyanova, Carola Bürgi

**Datum:** 15.06.2026

**Institution:** Ostschweizer Fachhochschule | Systemtechnik BSc | FS 2026

---

## 1. Einleitung

### 1.1 Aufgabenbeschreibung
Ziel dieses Projekts ist die Entwicklung und Implementierung eines Analytical Neural Networks (ANN) zur automatischen Klassifikation von Kleidungsstücken anhand von Bilddaten. Das Modell wird mit Hilfe der Bibliothek PyTorch trainiert und soll verschiedene Kleidungsarten zuverlässig erkennen und in die Kategorien Shorts, Schuhe, Hosen, Kleider und Shirts einteilen. Grundlage bildet ein realer Bilddatensatz aus einer öffentlich verfügbaren Quelle.

### 1.2 Anwendungsbereich
Die automatische Klassifikation von Kleidung ist ein typisches Problem im Bereich der Bildverarbeitung und des maschinellen Lernens. Praktische Anwendungen finden sich insbesondere im E-Commerce, wo Produkte automatisch kategorisiert und effizient verwaltet werden müssen.

Weitere Einsatzbereiche sind beispielsweise visuelle Suchsysteme, Empfehlungssysteme oder automatisierte Sortierprozesse in der Logistik. Auch in der Modeindustrie kann eine solche Technologie zur Analyse von Trends oder zur Unterstützung von Designprozessen eingesetzt werden.


### 1.3 Bedeutung des Themas
Durch dieses Projekt wird ein praktisches Verständnis für den Aufbau, das Training und die Optimierung von neuronalen Netzen entwickelt. Zudem zeigt es exemplarisch, wie theoretische Konzepte aus dem Bereich Machine Learning auf reale Datensätze angewendet werden können.

Die Bedeutung dieses Themas liegt in der zunehmenden Relevanz datengetriebener Systeme und künstlicher Intelligenz in industriellen und kommerziellen Anwendungen. Bildklassifikation mit neuronalen Netzen gehört zu den zentralen Aufgaben moderner KI-Systeme.

## 2. Zielsetzung und Vorgehensweise

### 2.1 Zielsetzung
Ziel dieses Projekts ist die Entwicklung eines leistungsfähigen Klassifikationsmodells für Kleidungsbilder auf Basis eines neuronalen Netzes. Dabei soll das vortrainierte Modell MobileNetV2 im Rahmen von Transfer Learning so angepasst werden, dass es die definierten Kategorien (Shorts, Schuhe, Hosen, Kleider, Shirts) möglichst zuverlässig erkennt.

Ein weiterer Fokus liegt auf der effizienten Nutzung vorhandener Rechenressourcen sowie auf der Reduktion der Trainingszeit, ohne dabei signifikante Einbussen bei der Modellgenauigkeit zu erzielen.

### 2.2 Adressierte Fragen
- Wie gut eignet sich Transfer Learning mit MobileNetV2 für die Klassifikation von Kleidungsbildern?
- Wie beeinflusst die Wahl von Hyperparametern die Performance des Modells?


### 2.3 Vorgehensweise
Zum Abschluss wird das trainierte Modell auf den Testdaten evaluiert und hinsichtlich Genauigkeit sowie Generalisierungsfähigkeit analysiert.

Der bereits gelabelte und strukturierte Datensatz wird direkt in Trainings-, Validierungs- und Testdaten aufgeteilt und für das Modelltraining vorbereitet.

Das Training erfolgt mittels Transfer Learning, wobei die vortrainierten Gewichte als Ausgangspunkt genutzt werden. Dabei werden ausgewählte Hyperparameter gezielt variiert, um die Modellleistung zu optimieren.

Anschliessend wird das vortrainierte Modell MobileNetV2 geladen und für die spezifische Klassifikationsaufgabe angepasst, indem die finale Klassifikationsschicht auf die vorhandenen Kategorien abgestimmt wird.

## 3. Datenladenoption und Preprocessing

### 3.1 Datenbeschreibung
Für dieses Projekt wird das Apparel Images Dataset verwendet, welches eine grosse Anzahl von Bildern unterschiedlicher Kleidungsstücke enthält. Die Bilder sind bereits kategorisiert und in klar definierte Klassen unterteilt, darunter Shorts, Schuhe, Hosen, Kleider und Shirts.

Der Datensatz zeichnet sich durch eine hohe Variabilität aus, da die Kleidungsstücke in unterschiedlichen Perspektiven, Farben und Formen dargestellt sind. Dadurch eignet er sich gut für die Anwendung von Deep Learning, da das Modell robuste visuelle Merkmale lernen muss, um die Klassen zuverlässig zu unterscheiden.

Die Daten liegen in strukturierter Form vor, sodass sie direkt für das Training eines neuronalen Netzes verwendet werden können.

### 3.2 Datenquellen
Der verwendete Datensatz stammt von der Plattform Kaggle und ist öffentlich zugänglich:
https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset/data

#### 3.2.1 Imortierung aller notwendigen Libraries

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import MobileNet_V2_Weights

# Check device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'pandas'

#### 3.2.2: Laden und erkunden des Datensatzes

In [ ]:
# Define the dataset path (customize this path to your local dataset location)
dataset_path = r'E:\daria\FS26\ANN\ANN_Project\project_ann\dataset'  # Change this to your dataset path

# Check if dataset exists
if os.path.exists(dataset_path):
    print(f"Dataset found at: {dataset_path}")
    print("\nDirectory structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{sub_indent}{file}')
        if len(files) > 3:
            print(f'{sub_indent}... and {len(files) - 3} more files')
else:
    print(f"Dataset not found at {dataset_path}")
    print("Please download the dataset from: https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset")
    print("And update the 'dataset_path' variable with the correct location.")

## 4. Modellaufbau und Beschreibung der Modell-Architektur

### 4.1 Netzwerk-Architektur
Für die Klassifikationsaufgabe wird das vortrainierte Convolutional Neural Network MobileNetV2 verwendet. Das Modell basiert auf einer tiefen CNN-Architektur, die speziell für effiziente Bildverarbeitung optimiert ist.

MobileNetV2 besteht aus mehreren Faltungsblöcken, die sogenannte Inverted Residual Blocks verwenden. Diese kombinieren Depthwise-Separable Convolutions mit linearen Bottlenecks, wodurch die Anzahl der Parameter und der Rechenaufwand reduziert werden.

Für die vorliegende Aufgabe wird das vortrainierte Modell übernommen und die finale Klassifikationsschicht ersetzt. Die neue Fully-Connected-Schicht wird so angepasst, dass sie die fünf Zielklassen (Shorts, Schuhe, Hosen, Kleider, Shirts) ausgibt.

### 4.2 Begründung der Architekturwahl
MobileNetV2 wurde gewählt, da es ein sehr gutes Verhältnis zwischen Modellgenauigkeit und Rechenaufwand bietet. Dies ist insbesondere relevant, da das Modell auf begrenzten Ressourcen (z. B. lokaler Rechner oder Colab) trainiert wird.

Durch die Verwendung von Transfer Learning können bereits gelernte Features aus dem ImageNet-Datensatz genutzt werden. Dadurch wird die Trainingszeit deutlich reduziert und gleichzeitig die Modellleistung verbessert, insbesondere im Vergleich zu einem von Grund auf trainierten Netzwerk.

### 4.3 Besonderheiten der Architektur
Eine zentrale Besonderheit von MobileNetV2 ist die Verwendung von Inverted Residual Blocks. Im Gegensatz zu klassischen CNNs wird dabei zunächst die Dimension erhöht und anschliessend wieder reduziert, was eine effizientere Feature-Extraktion ermöglicht.

Zusätzlich verwendet das Modell Depthwise-Separable Convolutions, bei denen räumliche und kanalweise Faltung getrennt durchgeführt werden. Dies reduziert den Rechenaufwand erheblich, ohne die Performance stark zu beeinträchtigen.

Ein weiterer Vorteil ist die kompakte Modellstruktur, die eine schnelle Trainings- und Inferenzzeit ermöglicht.

### 4.4 Mögliche Alternativen
Als Alternative zu MobileNetV2 könnten klassische Convolutional Neural Networks verwendet werden, die von Grund auf trainiert werden. Diese sind jedoch in der Regel rechenintensiver und benötigen mehr Trainingsdaten, um vergleichbare Ergebnisse zu erzielen.

Weitere mögliche Architekturen sind beispielsweise ResNet oder EfficientNet. Diese bieten teilweise höhere Genauigkeiten, gehen jedoch oft mit einem höheren Rechenaufwand einher.

## 8. Referenzen

### Quellen
- Vorlesungsunterlagen Modul ANN [Christoph Würsch]

---

## Eigenständigkeitserklärung

Hiermit bestätige ich / bestätigen wir, dass ich / wir die vorliegende Arbeit selbständig verfasst und keine anderen als die angegebenen Hilfsmittel benutzt haben. Die Stellen der Arbeit, die dem Wortlaut oder dem Sinn nach anderen Werken (dazu zählen auch Internetquellen) entnommen sind, wurden unter Angabe der Quelle kenntlich gemacht.

**Datum:** 14.06.2026

**Unterschrift(en):** 

Daria Lukyanova

Carola Bürgi